# MTB Utrechtse Heuvelrug: Jaarvergelijk 2026 vs 2025

**Activiteit:** MTB Utrechtse Heuvelrug (08 november 2026 en 09 november 2025)

**Vraag:** Hoe presteer ik (qua hartslag en snelheid) ten opzichte van een jaar geleden?

**Methode:**
1. Inlezen van de ruwe binaire FIT-data van beide ritten
2. Data opschonen en gaten door "Smart Recording" en Auto-Pause fixen
3. Rit opdelen in 25km kwadranten voor verval-analyse
4. Hartslagzones en verhouding beweegtijd/stilstand berekenen

**Resultaat:** een afbeelding in de map `output/`.

## Installatie van de benodigde pakketten

De pakketten staan in `requirements.txt`. Deze cel hoeft alleen de eerste keer na het opstarten van het cluster te draaien.

In [0]:
0# Herstart de Python-interpreter zodat een aangepaste module (zoals utils.grafieken)
# opnieuw wordt ingeleaden in plaats van uit de cache te komen.
%restart_python

# Let op: na de herstart zijn alle variabelen en imports weggeruimd —
# draai daarna de cellen 'Importeren functies' t/m 'Data inlezen' opnieuw.

In [0]:
%pip install -q -r ../requirements.txt

## Imports

In [0]:
from fitparse import FitFile
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
from utils.config import PADEN, ATLEET
from utils.visualisatie import exporteer_afbeelding
from utils.grafieken import teken_yoy_hartslag, teken_yoy_snelheid_en_tijd

## Instellingen

In [0]:
# Definieer de paden en labels
FIT_2025 = "../fit_bestanden_Garmin/2025/2025-11-09_08-33-00.fit"
LABEL_2025 = "2025 (Zeist/Austerlitz/Soest/Lage Vuursche/Hoge Vuursche)"

FIT_2026 = "../fit_bestanden_Garmin/2026/2026-11-08_09-13-36.fit"
LABEL_2026 = "2026 (Zeist/Leersum/Amerongen/Rhenen)"

## Data inlezen

In [0]:
def load_and_prep_fit(filepath, year_label):
    """Leest een FIT bestand en retourneert een DataFrame."""
    fitfile = FitFile(filepath)
    data = [{data_field.name: data_field.value for data_field in record} for record in fitfile.get_messages('record')]
        
    df = pd.DataFrame(data)
    speed_col = 'enhanced_speed' if 'enhanced_speed' in df.columns else 'speed'
    cols = [c for c in ['timestamp', 'heart_rate', speed_col, 'distance'] if c in df.columns]
    
    df = df.dropna(subset=['heart_rate', speed_col, 'distance']).copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Smart Recording & Pauze Fix
    df['time_diff'] = df['timestamp'].diff().dt.total_seconds().fillna(1)
    df['speed_kmh'] = df[speed_col] * 3.6
    df['distance_km'] = df['distance'] / 1000
    df['is_moving'] = df['speed_kmh'] > 3.0
    df.loc[df['time_diff'] > 30, 'is_moving'] = False
    
    # Labels en Bins toevoegen
    df['jaar'] = year_label
    df['kwadrant'] = pd.cut(df['distance_km'], bins=[0, 25, 50, 75, 100, 200], labels=['0-25 km', '25-50 km', '50-75 km', '75-100 km', '100+ km'])
    hr_dict = ATLEET.hr_zones
    HR_BINS = [val[0] for val in hr_dict.values()] + [list(hr_dict.values())[-1][1]]
    HR_LABELS = list(hr_dict.keys())
    df['hr_zone'] = pd.cut(df['heart_rate'], bins=HR_BINS, labels=HR_LABELS)
    
    return df

print("Bezig met inladen van FIT data...")
df_all = pd.concat([
    load_and_prep_fit(FIT_2025, LABEL_2025),
    load_and_prep_fit(FIT_2026, LABEL_2026)
])
df_moving = df_all[df_all['is_moving'] == True]
print("Data succesvol geladen en samengevoegd!")

## Berekening: Kernstatistieken

In [0]:
print("--- Kernstatistieken Overzicht ---")
for jaar in df_all['jaar'].unique():
    df_jaar = df_moving[df_moving['jaar'] == jaar]
    
    totale_afstand = df_jaar['distance_km'].max()
    totale_tijd_uren = df_jaar['time_diff'].sum() / 3600
    gem_snelheid = totale_afstand / totale_tijd_uren
    gem_hr = df_jaar['heart_rate'].mean()
    
    print(f"\n{jaar}:")
    print(f"  - Afstand:          {totale_afstand:.1f} km")
    print(f"  - Beweegtijd:       {totale_tijd_uren * 60:.0f} minuten")
    print(f"  - Gem. Snelheid:    {gem_snelheid:.1f} km/u")
    print(f"  - Gem. Hartslag:    {gem_hr:.0f} bpm")

## Grafiek

In [0]:
TITEL_HARTSLAG = "MTB UTRECHTSE HEUVELRUG: HARTSLAGANALYSE (2025 VS. 2026)"
fig_hr = teken_yoy_hartslag(
    df_moving=df_moving, 
    label_2025=LABEL_2025, 
    label_2026=LABEL_2026, 
    titel=TITEL_HARTSLAG
)

In [0]:
TITEL_PACING = "MTB UTRECHTSE HEUVELRUG: PACING & TOTALE TIJD (2025 VS. 2026)"
fig_pacing = teken_yoy_snelheid_en_tijd(
    df_all=df_all, 
    df_moving=df_moving, 
    label_2025=LABEL_2025, 
    label_2026=LABEL_2026, 
    titel=TITEL_PACING
)

## Exporteren

In [0]:
exporteer_afbeelding(fig_hr, bestandsnaam="20261108_mtb_yoy_hartslag", formaat="liggend")
exporteer_afbeelding(fig_pacing, bestandsnaam="20261108_mtb_yoy_pacing_en_tijd", formaat="liggend")